In [ ]:
!pip install langchain langchain-openai langchain-community python-dotenv

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 실습 하나 : LLM 추상화 - 복잡한 로직을 숨기고 실행을 단순화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature="0.7")
response = llm.invoke("장마철 건강관리는 어떻게 해?")
print(response.content)


In [ ]:
from pathlib import PureWindowsPath
# PromptTemplate 사용 : 변수 삽입 로직 추상화
template = """
당신은 한국어 전문가입니다.
아래 주제로 아름다운 노래 작사를 만들어 줘.

주제:
"{content}"
"""
# print(template)

prompt = PromptTemplate(
    input_variables=["content"],
    template = template
)

filled_prompt = prompt.format(
    content="소나기"
)

print('filled_prompt : ', filled_prompt)
response = llm.invoke(filled_prompt)
print(response.content)

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
# Tool 작성 및 호출

@tool
def calculator(expression:str) -> str:
    """
    간단한 사칙연산을 하는 계산 기능 (docstring 이런 문장 필수)
    """
    try:
        result = eval(expression)
        return f'{expression} = {result}'
    except Exception as e:
        return f'계산 실패 : {e}'

tools = [calculator]   # Agent가 사용할 수 있는 도구(tool)목록

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="계산이 필요하면 calculator 도구를 사용해"
)

result = agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"7 * (3 + 2) / 2는 얼마야"
        }  # LLM이 직접 계산하지 않고, tool호출
    ]
})

print(result["messages"][-1].content)
